In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

import plotly.express as px
import joblib

In [3]:
df = pd.read_csv("../data/raw/train.csv").copy()
df.head()

,customer_id,age,location,subscription_type,payment_plan,num_subscription_pauses,payment_method,customer_service_inquiries,signup_date,weekly_hours,average_session_length,song_skip_rate,weekly_songs_played,weekly_unique_songs,num_favorite_artists,num_platform_friends,num_playlists_created,num_shared_playlists,notifications_clicked,churned
0,1,32,Montana,Free,Yearly,2,Paypal,Medium,-1606,22.391362,105.394516,0.176873,169,109,18,32,52,35,46,0
1,2,64,New Jersey,Free,Monthly,3,Paypal,Low,-2897,29.294210,52.501115,0.981811,55,163,44,33,12,25,37,1
2,3,51,Washington,Premium,Yearly,2,Credit Card,High,-348,15.400312,24.703696,0.048411,244,117,20,129,50,28,38,0
3,4,63,California,Family,Yearly,4,Apple Pay,Medium,-2894,22.842084,83.595480,0.035691,442,252,47,120,55,17,24,0
4,5,54,Washington,Family,Monthly,3,Paypal,High,-92,23.151163,52.578266,0.039738,243,230,41,66,40,32,47,0


In [4]:
TARGET = "churned"

In [5]:
DROP_COLS = ["customer_id"]  # ajusta el nombre si es distinto

Añadimos variables interesantes a nuestro modelo(ya hecho en EDA)

In [6]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # signup_date en el dataset parece funcionar mejor como antigüedad
    if "signup_date" in df.columns:
        df["tenure_days"] = -df["signup_date"]
    
    # grupos de edad
    if "age" in df.columns:
        df["age_group"] = pd.cut(
            df["age"],
            bins=[18, 25, 35, 50, 65, 80],
            labels=["18-24", "25-34", "35-49", "50-64", "65-79"],
            include_lowest=True
        )
    
    # bins de uso semanal
    if "weekly_hours" in df.columns:
        df["weekly_hours_bin"] = pd.cut(
            df["weekly_hours"],
            bins=[0, 5, 10, 20, 30, 40, 50],
            labels=["0-5", "5-10", "10-20", "20-30", "30-40", "40-50"],
            include_lowest=True
        )
    
    # bins de skip rate
    if "song_skip_rate" in df.columns:
        df["skip_rate_bin"] = pd.cut(
            df["song_skip_rate"],
            bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
            labels=["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"],
            include_lowest=True
        )
    
    # segmentación simple de riesgo interpretativa
    if {"subscription_type", "customer_service_inquiries"}.issubset(df.columns):
        conditions = [
            (df["subscription_type"] == "Free") & (df["customer_service_inquiries"] == "High"),
            (df["subscription_type"] == "Free"),
            (df["customer_service_inquiries"] == "High")
        ]
        choices = ["Very High Risk Segment", "High Risk Segment", "Medium Risk Segment"]
        df["risk_segment_rule"] = np.select(conditions, choices, default="Lower Risk Segment")
    
    return df

In [7]:
df_model = add_features(df)
df_model.head()

,customer_id,age,location,subscription_type,payment_plan,num_subscription_pauses,payment_method,customer_service_inquiries,signup_date,weekly_hours,...,num_platform_friends,num_playlists_created,num_shared_playlists,notifications_clicked,churned,tenure_days,age_group,weekly_hours_bin,skip_rate_bin,risk_segment_rule
0,1,32,Montana,Free,Yearly,2,Paypal,Medium,-1606,22.391362,...,32,52,35,46,0,1606,25-34,20-30,0-0.2,High Risk Segment
1,2,64,New Jersey,Free,Monthly,3,Paypal,Low,-2897,29.294210,...,33,12,25,37,1,2897,50-64,20-30,0.8-1.0,High Risk Segment
2,3,51,Washington,Premium,Yearly,2,Credit Card,High,-348,15.400312,...,129,50,28,38,0,348,50-64,10-20,0-0.2,Medium Risk Segment
3,4,63,California,Family,Yearly,4,Apple Pay,Medium,-2894,22.842084,...,120,55,17,24,0,2894,50-64,20-30,0-0.2,Lower Risk Segment
4,5,54,Washington,Family,Monthly,3,Paypal,High,-92,23.151163,...,66,40,32,47,0,92,50-64,20-30,0-0.2,Medium Risk Segment


## Selección de variables para el modelo

In [8]:
feature_cols = [col for col in df_model.columns if col != TARGET and col not in DROP_COLS]

X = df_model[feature_cols].copy()
y = df_model[TARGET].copy()

print("Número de features:", len(feature_cols))
print("Shape X:", X.shape)
print("Shape y:", y.shape)

Número de features: 23
Shape X: (125000, 23)
Shape y: (125000,)


Separamos las numéricas y categóricas

In [9]:
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numéricas:", numeric_features)
print("Categóricas:", categorical_features)

Numéricas: ['age', 'num_subscription_pauses', 'signup_date', 'weekly_hours', 'average_session_length', 'song_skip_rate', 'weekly_songs_played', 'weekly_unique_songs', 'num_favorite_artists', 'num_platform_friends', 'num_playlists_created', 'num_shared_playlists', 'notifications_clicked', 'tenure_days']
Categóricas: ['location', 'subscription_type', 'payment_plan', 'payment_method', 'customer_service_inquiries', 'age_group', 'weekly_hours_bin', 'skip_rate_bin', 'risk_segment_rule']


/var/folders/c5/zwyc201564b7l_drzhj5njy00000gn/T/ipykernel_66340/735017401.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()


## Preprocesado usando Pipeline

In [16]:
from sklearn.preprocessing import StandardScaler

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

La regresión logística inicialmente mostró un problema de convergencia, algo habitual cuando las variables numéricas no están escaladas. Para solucionarlo, incorporé estandarización en el pipeline y aumenté el nº máximo de iteraciones

## Split entrenamiento/test

In [17]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_valid.shape)

(100000, 23) (25000, 23)


## Regresión logística

In [18]:
logreg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=3000, random_state=42))
])

logreg_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

## Función de evaluación

In [19]:
def evaluate_model(model, X_valid, y_valid, model_name="model"):
    y_pred = model.predict(X_valid)
    y_proba = model.predict_proba(X_valid)[:, 1]
    
    metrics = pd.DataFrame({
        "model": [model_name],
        "accuracy": [accuracy_score(y_valid, y_pred)],
        "precision": [precision_score(y_valid, y_pred)],
        "recall": [recall_score(y_valid, y_pred)],
        "f1": [f1_score(y_valid, y_pred)],
        "roc_auc": [roc_auc_score(y_valid, y_proba)]
    })
    
    cm = confusion_matrix(y_valid, y_pred)
    
    preds = X_valid.copy()
    preds["y_true"] = y_valid.values
    preds["y_pred"] = y_pred
    preds["p_churn"] = y_proba
    
    return metrics, cm, preds

In [20]:
logreg_metrics, logreg_cm, logreg_preds = evaluate_model(
    logreg_model,
    X_valid,
    y_valid,
    model_name="Logistic Regression"
)

logreg_metrics

,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.83664,0.841329,0.84028,0.840805,0.926266


La regresión logística se usó como baselina interpretable y obtuvo un rendimiento sólido y equilibrado, con métricas consistentes entre precisión, recall y F1, además de un ROC AUC superior a 0.92

## Modelo de Random Forest

In [15]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_metrics, rf_cm, rf_preds = evaluate_model(
    rf_model,
    X_valid,
    y_valid,
    model_name="Random Forest"
)

rf_metrics

,model,accuracy,precision,recall,f1,roc_auc
0,Random Forest,0.84728,0.846514,0.858122,0.852279,0.937824


## Comparación de modelos

In [21]:
metrics_df = pd.concat([logreg_metrics, rf_metrics], ignore_index=True)
metrics_df

,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.83664,0.841329,0.840280,0.840805,0.926266
1,Random Forest,0.84728,0.846514,0.858122,0.852279,0.937824


Gráfico de comparación

In [22]:
metrics_long = metrics_df.melt(
    id_vars="model",
    var_name="metric",
    value_name="score"
)

fig = px.bar(
    metrics_long,
    x="metric",
    y="score",
    color="model",
    barmode="group",
    title="Comparación de modelos"
)

fig.show()

Se probó un Random Forest para captar relaciones no lineales e interacciones entre variables. Este segundo modelo mejoró el rendimiento en todas las métricas principales, especialmente en recall, F1 y ROC AUC, por lo que se seleccionó como modelo final

## Matriz de confusión

In [31]:
from sklearn.metrics import confusion_matrix
import pandas as pd
import plotly.express as px

def plot_confusion_matrix_percentage(y_true, y_pred, model_name="Modelo"):
    """
    Muestra una matriz de confusión normalizada por fila (en porcentaje).
    Cada fila suma 100%.
    """
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    cm_df = pd.DataFrame(
        cm_pct,
        index=["Real: No Churn", "Real: Churn"],
        columns=["Predicho: No Churn", "Predicho: Churn"]
    )

    fig = px.imshow(
        cm_df,
        text_auto=".2f",
        aspect="auto",
        color_continuous_scale="Plasma",
        title=f"Matriz de confusión normalizada (%) - {model_name}",
        labels={"color": "Porcentaje"}
    )

    fig.update_traces(
        texttemplate="%{z:.2f}%",
        textfont_size=16
    )

    fig.update_layout(
        xaxis_title="Clase predicha",
        yaxis_title="Clase real",
        title_x=0.5
    )

    fig.show()

    return cm_df

In [35]:
y_pred_rf = rf_model.predict(X_valid)
rf_cm_pct_df = plot_confusion_matrix_percentage(y_valid, y_pred_rf, "Random Forest")
rf_cm_pct_df

,Predicho: No Churn,Predicho: Churn
Real: No Churn,83.584053,16.415947
Real: Churn,14.187768,85.812232


In [37]:
y_pred_logreg = logreg_model.predict(X_valid)
logreg_cm_pct_df = plot_confusion_matrix_percentage(y_valid, y_pred_logreg, "Logistic Regression")
logreg_cm_pct_df

,Predicho: No Churn,Predicho: Churn
Real: No Churn,83.279901,16.720099
Real: Churn,15.971952,84.028048


Guardamos las matrices de confusión

In [39]:
rf_cm_pct_df.to_csv("../data/exports/rf_confusion_matrix_percentage.csv")
logreg_cm_pct_df.to_csv("../data/exports/logreg_confusion_matrix_percentage.csv")

## Guardar los artefactos para Streamlit

In [51]:
metrics_df.to_csv("../data/exports/model_metrics.csv", index=False)
logreg_preds.to_csv("../data/exports/logreg_validation_predictions.csv", index=False)
rf_preds.to_csv("../data/exports/rf_validation_predictions.csv", index=False)

joblib.dump(rf_model, "../models/rf_model.joblib")
joblib.dump(logreg_model, "../models/logreg_model.joblib")

['../models/logreg_model.joblib']

Guardamos el dataset con las variables creadas

In [52]:
df_model.to_csv("../data/processed/train_model_ready.csv", index=False)

## Variables más importantes para el modelo

In [41]:
preprocessor_fitted = rf_model.named_steps["preprocessor"]
rf_classifier = rf_model.named_steps["classifier"]

Volvemos a las variables originales

In [42]:
feature_names = preprocessor_fitted.get_feature_names_out()
len(feature_names)

66

Tabla de importancia de las variables

In [46]:
import pandas as pd

importances_df["feature_clean"] = (
    importances_df["feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
    .str.replace("_", " ", regex=False)
)

importances_df.head(20)

,feature,importance,feature_clean
3,num__weekly_hours,0.095288,weekly hours
63,cat__risk_segment_rule_Lower Risk Segment,0.070727,risk segment rule Lower Risk Segment
5,num__song_skip_rate,0.057056,song skip rate
1,num__num_subscription_pauses,0.056077,num subscription pauses
34,cat__subscription_type_Free,0.042778,subscription type Free
0,num__age,0.041757,age
44,cat__customer_service_inquiries_Low,0.041496,customer service inquiries Low
43,cat__customer_service_inquiries_High,0.038318,customer service inquiries High
51,cat__weekly_hours_bin_0-5,0.034432,weekly hours bin 0-5
55,cat__weekly_hours_bin_40-50,0.031377,weekly hours bin 40-50


Gráfico enseñando las más importantes

In [47]:
import plotly.express as px

top_n = 15
top_importances = importances_df.head(top_n).sort_values("importance", ascending=True)

fig = px.bar(
    top_importances,
    x="importance",
    y="feature",
    orientation="h",
    title=f"Top {top_n} variables más importantes - Random Forest",
    text="importance"
)

fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(
    xaxis_title="Importancia",
    yaxis_title="Variable",
    title_x=0.5
)

fig.show()

Guardar las tablas para streamlit

In [48]:
importances_df.to_csv("../data/exports/rf_feature_importance.csv", index=False)

### Interpretación y comparació con EDA

Las **variables más relevantes** se agrupan en tres grandes bloques.<br>

- El primero es el **engagement del usuario**. Aquí destacan especialmente weekly_hours, song_skip_rate, el número de canciones escuchadas por semana, la duración media de sesión y también variables como las notificaciones clicadas. Esto sugiere que el nivel de uso y la calidad de la interacción con la plataforma son muy importantes para explicar el abandono.<br>

- El segundo bloque tiene que ver con la **fricción o inestabilidad del usuario**. Aquí aparecen variables como num_subscription_pauses y customer_service_inquiries. Esto refuerza la idea de que los usuarios que tienen más incidencias, más pausas o señales de comportamiento inestable presentan un mayor riesgo de churn.<br>

- El tercer bloque corresponde al **tipo de cliente**. Variables como subscription_type_Free o subscription_type_Student aparecen entre las más importantes, lo que indica que el tipo de suscripción también condiciona claramente la retención.<br>

Además, añadí una variable derivada llamada risk_segment_rule, que resume ciertas combinaciones de riesgo detectadas durante el análisis exploratorio. Esta variable también aparece entre las más influyentes, lo que sugiere que incorporar conocimiento del negocio en el feature engineering puede mejorar la capacidad predictiva del modelo.<br>

**En general**, la importancia de variables confirma bastante bien lo que ya se observaba en el EDA: **el churn parece estar impulsado** sobre todo por **tipo de usuario**, **engagement** y **señales de fricción**.